# 🍎 APPLE CNN — MASTER MODEL TRAINING
### Project: Plant Disease Detection (Computer Vision)
**Architecture:** MobileNetV2 (Transfer Learning) + Data Augmentation  
**Dataset:** `3rd Preprocessing/apple_processed_data.npz` (Internal PlantVillage + External Plant Pathology 2021)  
**Target Classes (3):** `Healthy (0)`, `Apple_Scab (1)`, `Cedar_Apple_Rust (2)`

In [3]:
# ================================================================
# 🍎 APPLE — FINAL MODEL TRAINING PRE-CHECK
# READ-ONLY AUDIT
# ================================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import numpy as np

# ------------------------------------------------
# PROJECT PATH
# ------------------------------------------------
BASE = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)"
if not os.path.exists(BASE):
    BASE = r"G:\My Drive\Plant Disease Detection (Computer Vision)"

PREPROCESS = os.path.join(BASE, "3rd Preprocessing")
TRAINING   = os.path.join(BASE, "4th Model_Training")
MODEL_DIR  = os.path.join(BASE, "6th Trained_Model")

NPZ_PATH   = os.path.join(PREPROCESS, "apple_processed_data.npz")

print("=" * 75)
print("🍎 APPLE DATASET & ENVIRONMENT AUDIT")
print("=" * 75)
print("NPZ Path:", NPZ_PATH)
if os.path.exists(NPZ_PATH):
    data = np.load(NPZ_PATH)
    print("✅ NPZ File Found! Size:", round(os.path.getsize(NPZ_PATH)/(1024**2), 2), "MB")
    print("Arrays in NPZ:", data.files)
    print("X shape:", data['X'].shape, "dtype:", data['X'].dtype)
    print("y shape:", data['y'].shape, "dtype:", data['y'].dtype)
    print("Classes:", data['class_names'])
else:
    print("❌ NPZ File Not Found!")


Exception ignored in: <function NpzFile.__del__ at 0x783a8bb24a40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 228, in __del__
    self.close()
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 223, in close
    self.fid.close()
OSError: [Errno 107] Transport endpoint is not connected


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🍎 APPLE DATASET & ENVIRONMENT AUDIT
NPZ Path: /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/3rd Preprocessing/apple_processed_data.npz
✅ NPZ File Found! Size: 201.98 MB
Arrays in NPZ: ['X', 'y', 'class_names', 'source']
X shape: (1775, 224, 224, 3) dtype: uint8
y shape: (1775,) dtype: int64
Classes: ['Healthy' 'Apple_Scab' 'Cedar_Apple_Rust']


In [4]:
# ================================================================
# 🍎 APPLE CNN — FINAL MASTER TRAINING PIPELINE
# Transfer Learning via MobileNetV2 + Data Augmentation
# ================================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
import csv
import random
import hashlib
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split

# ================================================================
# 1. REPRODUCIBILITY SEED
# ================================================================

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# ================================================================
# 2. PATHS & ARTIFACTS
# ================================================================

BASE = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)"
if not os.path.exists(BASE):
    BASE = r"G:\My Drive\Plant Disease Detection (Computer Vision)"

PREPROCESS_ROOT = os.path.join(BASE, "3rd Preprocessing")
TRAINING_ROOT   = os.path.join(BASE, "4th Model_Training")
MODEL_ROOT      = os.path.join(BASE, "6th Trained_Model")

os.makedirs(TRAINING_ROOT, exist_ok=True)
os.makedirs(MODEL_ROOT, exist_ok=True)

NPZ_PATH     = os.path.join(PREPROCESS_ROOT, "apple_processed_data.npz")
MODEL_PATH   = os.path.join(MODEL_ROOT, "apple_cnn_best.keras")
HISTORY_PATH = os.path.join(TRAINING_ROOT, "apple_CNN_training_history.npz")
LOG_PATH     = os.path.join(TRAINING_ROOT, "apple_CNN_training_log.csv")
SUMMARY_PATH = os.path.join(TRAINING_ROOT, "apple_CNN_training_summary.json")

print("=" * 75)
print("🍎 1. LOADING APPLE DATASET")
print("=" * 75)

data = np.load(NPZ_PATH, allow_pickle=False)
X = data['X']
y = data['y']
class_names = data['class_names']
source = data['source']

print(f"Total samples : {len(X)}")
print(f"X shape       : {X.shape}, dtype: {X.dtype}")
print(f"y shape       : {y.shape}, dtype: {y.dtype}")
print(f"Classes       : {class_names.tolist()}")

# ================================================================
# 3. STRATIFIED DATASET SPLIT (70% Train, 15% Val, 15% Test)
# ================================================================

print("\n" + "=" * 75)
print("🍎 2. STRATIFIED TRAIN / VAL / TEST SPLIT")
print("=" * 75)

# First split: 70% Train, 30% Temp (Val + Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=SEED,
    stratify=y
)

# Second split: 15% Val, 15% Test from Temp
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print(f"Train Set : {X_train.shape[0]} images ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Val Set   : {X_val.shape[0]} images ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test Set  : {X_test.shape[0]} images ({X_test.shape[0]/len(X)*100:.1f}%)")

# ================================================================
# 4. MODEL ARCHITECTURE (MobileNetV2 Transfer Learning)
# ================================================================

print("\n" + "=" * 75)
print("🍎 3. BUILDING MOBILENETV2 MODEL")
print("=" * 75)

IMG_SHAPE = (224, 224, 3)
NUM_CLASSES = len(class_names)

# In-model data augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2)
], name="apple_augmentation")

# Base pre-trained model
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SHAPE,
    include_top=False,
    weights="imagenet"
)
base_model.trainable = True

# Fine-tune from layer 100 onwards
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

inputs = tf.keras.Input(shape=IMG_SHAPE)
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs, name="Apple_MobileNetV2")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ================================================================
# 5. TRAINING WITH CALLBACKS
# ================================================================

print("\n" + "=" * 75)
print("🍎 4. STARTING MODEL TRAINING")
print("=" * 75)

EPOCHS = 25
BATCH_SIZE = 32

cb_list = [
    callbacks.ModelCheckpoint(
        MODEL_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),
    callbacks.CSVLogger(LOG_PATH)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=cb_list,
    verbose=1
)

# Save history array
np.savez_compressed(
    HISTORY_PATH,
    loss=history.history['loss'],
    accuracy=history.history['accuracy'],
    val_loss=history.history['val_loss'],
    val_accuracy=history.history['val_accuracy']
)

# ================================================================
# 6. SAVE TRAINING SUMMARY JSON
# ================================================================

best_val_acc = float(np.max(history.history['val_accuracy']))
final_train_acc = float(history.history['accuracy'][-1])

summary_data = {
    "project": "Plant Disease Detection",
    "plant": "Apple",
    "model": "Apple CNN — MobileNetV2",
    "model_path": MODEL_PATH,
    "dataset_path": NPZ_PATH,
    "total_images": len(X),
    "train_images": len(X_train),
    "val_images": len(X_val),
    "test_images": len(X_test),
    "classes": class_names.tolist(),
    "best_val_accuracy": best_val_acc,
    "best_val_accuracy_percent": best_val_acc * 100,
    "epochs_trained": len(history.history['loss']),
    "seed": SEED
}

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=4)

print("\n" + "=" * 75)
print("✅ APPLE MODEL TRAINING COMPLETE")
print("=" * 75)
print(f"Best Validation Accuracy : {best_val_acc*100:.2f}%")
print(f"Model saved at           : {MODEL_PATH}")
print(f"Log saved at             : {LOG_PATH}")
print(f"Summary saved at         : {SUMMARY_PATH}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🍎 1. LOADING APPLE DATASET
Total samples : 1775
X shape       : (1775, 224, 224, 3), dtype: uint8
y shape       : (1775,), dtype: int64
Classes       : ['Healthy', 'Apple_Scab', 'Cedar_Apple_Rust']

🍎 2. STRATIFIED TRAIN / VAL / TEST SPLIT
Train Set : 1242 images (70.0%)
Val Set   : 266 images (15.0%)
Test Set  : 267 images (15.0%)

🍎 3. BUILDING MOBILENETV2 MODEL
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "Apple_MobileNetV2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ apple_augmentation (Sequential) │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,339 (9.24 MB)

 Trainable params: 2,025,795 (7.73 MB)

 Non-trainable params: 396,544 (1.51 MB)


🍎 4. STARTING MODEL TRAINING
Epoch 1/25
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.5355 - loss: 1.0029
Epoch 1: val_accuracy improved from None to 0.81203, saving model to /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/6th Trained_Model/apple_cnn_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/6th Trained_Model/apple_cnn_best.keras
39/39 ━━━━━━━━━━━━━━━━━━━━ 172s 4s/step - accuracy: 0.6731 - loss: 0.7480 - val_accuracy: 0.8120 - val_loss: 0.4990 - learning_rate: 1.0000e-04
Epoch 2/25
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8882 - loss: 0.3263
Epoch 2: val_accuracy improved from 0.81203 to 0.89098, saving model to /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/6th Trained_Model/apple_cnn_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/6th Trained_Model/apple_cnn_best.keras
39/39 ━━━━━━━━━━━━━━━━━━━━ 138s 3